# 08 · Command-line interface

`python -m pyterraplot` renders any xarray-readable file (`.nc`, `.zarr`, `.grib2`, `.h5`, …) to a
self-contained HTML — no Python scripting needed. It wraps the same accessor methods:
`to_html`, `frames_to_html`, `quiver_html`, `compare_html`.

The `!` lines below shell out from the notebook. First we write a sample NetCDF to feed the CLI.

In [ ]:
import numpy as np, xarray as xr

nlat, nlon, ntime = 49, 96, 6
lats = np.linspace(80, -80, nlat); lons = np.linspace(-180, 180, nlon)
LON, LAT = np.meshgrid(lons, lats)
base = 8*np.cos(np.radians(LAT))*np.sin(np.radians(2*LON))

t2m = xr.DataArray(
    np.stack([base*np.cos(p) for p in np.linspace(0, 2*np.pi, ntime, endpoint=False)]).astype("float32"),
    dims=["time", "lat", "lon"],
    coords={"time": np.arange(ntime), "lat": lats, "lon": lons},
    name="t2m", attrs={"units": "K", "long_name": "demo temperature"},
)
u = xr.DataArray((-np.sin(np.radians(LAT))).astype("float32")*10, dims=["lat","lon"],
                 coords={"lat":lats,"lon":lons}, name="u10", attrs={"units":"m/s"})
v = xr.DataArray(( np.cos(np.radians(LON))).astype("float32")*10, dims=["lat","lon"],
                 coords={"lat":lats,"lon":lons}, name="v10", attrs={"units":"m/s"})
xr.Dataset({"t2m": t2m, "u10": u, "v10": v}).to_netcdf("cli_demo.nc")
print("wrote cli_demo.nc")

## Help

In [ ]:
!python -m pyterraplot --help

## Single field → globe HTML

Extra non-lat/lon dims are auto-`isel`'d to index 0 (pass `--isel time=N` to pick another).

In [ ]:
!python -m pyterraplot cli_demo.nc --var t2m --out cli_globe.html --cmap RdYlBu_r

## 2D projection

In [ ]:
!python -m pyterraplot cli_demo.nc --var t2m --projection naturalEarth --out cli_map.html --cmap viridis

## Animate over a dimension

In [ ]:
!python -m pyterraplot cli_demo.nc --var t2m --animate time --interval 500 --out cli_anim.html

## Quiver (`--u`/`--v`, optional `--bg`)

In [ ]:
!python -m pyterraplot cli_demo.nc --u u10 --v v10 --bg t2m --out cli_winds.html

## Compare two variables

In [ ]:
!python -m pyterraplot cli_demo.nc --compare u10 v10 --symmetric --out cli_compare.html

In [ ]:
from pathlib import Path
for f in ["cli_globe.html","cli_map.html","cli_anim.html","cli_winds.html","cli_compare.html"]:
    if Path(f).exists():
        print(f"{f:20s} {Path(f).stat().st_size/1024:6.0f} kB")